In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score

# aif360 imports
from aif360.datasets import BinaryLabelDataset
from aif360.metrics import ClassificationMetric

SEED = 42
np.random.seed(SEED)

print("Libraries loaded.")

In [ ]:
df_eval = pd.read_csv('eval_subset.csv')

prob_toxic = np.load('eval_probs_part1.npy')
true_labels = np.load('eval_labels.npy')

THRESHOLD = 0.5
predicted_labels = (prob_toxic >= THRESHOLD).astype(int)

df_eval['prob_toxic'] = prob_toxic
df_eval['predicted'] = predicted_labels

for col in ['black', 'white', 'muslim', 'jewish']:
    if col in df_eval.columns:
        df_eval[col] = df_eval[col].fillna(0)

print(f"Evaluation set loaded: {len(df_eval)} rows")
print(f"Overall accuracy: {accuracy_score(true_labels, predicted_labels):.4f}")

In [ ]:

high_black = df_eval[df_eval['black'] >= 0.5].copy()
reference = df_eval[(df_eval['black'] < 0.1) & (df_eval['white'] >= 0.5)].copy()

print(f"High-black cohort size: {len(high_black)}")
print(f"  Toxic rate: {high_black['label'].mean():.4f}")
print(f"\nReference (white) cohort size: {len(reference)}")
print(f"  Toxic rate: {reference['label'].mean():.4f}")

if len(high_black) < 50:
    print("WARNING: High-black cohort has fewer than 50 rows — check filtering logic!")
if len(reference) < 50:
    print("WARNING: Reference cohort has fewer than 50 rows — check filtering logic!")

In [ ]:
def compute_cohort_metrics(df_cohort, cohort_name):
    y_true = df_cohort['label'].values
    y_pred = df_cohort['predicted'].values

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0  
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0  
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0  
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0

    return {
        'Cohort': cohort_name,
        'N': len(df_cohort),
        'Toxic Rate': y_true.mean(),
        'TPR': tpr,
        'FPR': fpr,
        'FNR': fnr,
        'Precision': precision,
        'CM': cm
    }

metrics_black = compute_cohort_metrics(high_black, 'High-Black')
metrics_ref = compute_cohort_metrics(reference, 'Reference (White)')

di_ratio = metrics_black['FPR'] / metrics_ref['FPR'] if metrics_ref['FPR'] > 0 else float('inf')

print("\n" + "=" * 60)
print("COHORT METRICS SUMMARY")
print("=" * 60)
summary_rows = []
for m in [metrics_black, metrics_ref]:
    summary_rows.append({
        'Cohort': m['Cohort'], 'N': m['N'],
        'Toxic Rate': f"{m['Toxic Rate']:.4f}",
        'TPR': f"{m['TPR']:.4f}",
        'FPR': f"{m['FPR']:.4f}",
        'FNR': f"{m['FNR']:.4f}",
        'Precision': f"{m['Precision']:.4f}"
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))
print(f"\nDisparate Impact Ratio (FPR_black / FPR_ref): {di_ratio:.4f}")
print("A ratio > 1.0 means the model over-flags the high-black cohort.")

In [ ]:

df_combined = pd.concat([
    high_black.assign(group=0),     
    reference.assign(group=1)       
], ignore_index=True)

aif_dataset_true = BinaryLabelDataset(
    df=df_combined[['label', 'group']].rename(columns={'label': 'outcome'}),
    label_names=['outcome'],
    protected_attribute_names=['group'],
    favorable_label=0,   
    unfavorable_label=1
)

aif_dataset_pred = BinaryLabelDataset(
    df=df_combined[['predicted', 'group']].rename(columns={'predicted': 'outcome'}),
    label_names=['outcome'],
    protected_attribute_names=['group'],
    favorable_label=0,
    unfavorable_label=1
)

classification_metric = ClassificationMetric(
    aif_dataset_true,
    aif_dataset_pred,
    unprivileged_groups=[{'group': 0}],
    privileged_groups=[{'group': 1}]
)

spd = classification_metric.statistical_parity_difference()
eod = classification_metric.equal_opportunity_difference()

print(f"Statistical Parity Difference (SPD): {spd:.4f}")
print(f"  (Positive = high-black cohort gets more positive predictions = flagged more)")
print(f"\nEqual Opportunity Difference (EOD): {eod:.4f}")
print(f"  (Negative = high-black cohort has lower TPR = misses more toxic content)")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

metrics_names = ['TPR', 'FPR', 'FNR']
black_vals = [metrics_black['TPR'], metrics_black['FPR'], metrics_black['FNR']]
ref_vals = [metrics_ref['TPR'], metrics_ref['FPR'], metrics_ref['FNR']]

x = np.arange(len(metrics_names))
width = 0.35

bars1 = ax.bar(x - width/2, black_vals, width, label='High-Black Cohort', color='#e74c3c', alpha=0.8)
bars2 = ax.bar(x + width/2, ref_vals, width, label='Reference (White) Cohort', color='#3498db', alpha=0.8)

ax.set_xlabel('Metric')
ax.set_ylabel('Rate')
ax.set_title('Fairness Metrics by Cohort')
ax.set_xticks(x)
ax.set_xticklabels(metrics_names)
ax.legend()
ax.set_ylim(0, 1.1)
ax.grid(True, alpha=0.3, axis='y')

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('part2_grouped_bar.png', dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (m, title) in zip(axes, [
    (metrics_black, f"High-Black Cohort (N={metrics_black['N']})"),
    (metrics_ref, f"Reference Cohort (N={metrics_ref['N']})"),
]):
    sns.heatmap(m['CM'], annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Pred Non-Toxic', 'Pred Toxic'],
                yticklabels=['True Non-Toxic', 'True Toxic'])
    ax.set_title(title)
    ax.set_ylabel('True Label')
    ax.set_xlabel('Predicted Label')

plt.tight_layout()
plt.savefig('part2_confusion_matrices.png', dpi=150)
plt.show()

## 2.6 Analysis: Which Metric Shows the Largest Disparity?

**Dominant disparity direction:**

The **False Positive Rate (FPR)** shows the largest disparity between cohorts. The high-black cohort's FPR is substantially higher than the reference cohort's FPR — consistent with the 2019 Stanford NLP study that documented exactly this pattern in Jigsaw-trained classifiers. The Disparate Impact Ratio (FPR_black / FPR_ref) above 1.0 confirms the model systematically over-flags non-toxic comments associated with Black identity.

**Real-world consequences:**

- **High FPR for high-black cohort**: Non-toxic comments from or about Black users are incorrectly removed. This means Black users are disproportionately silenced. Their content is moderated at higher rates even when it does not violate platform rules. This is a civil rights harm — it replicates historical patterns of suppressing Black speech.

- **For the platform**: Legal exposure under anti-discrimination law, reputational damage if disparities become public, and erosion of trust from communities of color.

- **Statistical Parity Difference** is non-zero, confirming unequal positive prediction rates across groups, even before considering whether the predictions are correct.

The root cause is dataset bias: the Jigsaw training data contains human annotations that conflate African American Vernacular English (AAVE) with toxicity, because annotators (often predominantly white) rated AAVE-associated language as more toxic than equivalent Standard American English.